# 02 standardize metadata

**Purpose:** Standardize sample, patient, dataset, and condition metadata.

**Original analysis notebook:** `02_attach_metadata_to_merged_fullgene_corrected.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# 02 — Metadata audit + standardization for merged 4-dataset full-gene object

This is the corrected metadata step for the restarted 4-dataset core pipeline.

**Input full-gene merged object**

`${GLIOMA_PROJECT_ROOT}/processed/integrated/GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad`

**Main output**

`${GLIOMA_PROJECT_ROOT}/processed/integrated/GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad`

This step does **not** normalize, subset genes, run HVG, run scVI, or change raw counts.  
It only standardizes `adata.obs` metadata and writes audit reports.

In [ ]:
import os
import gc
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

# -------------------------
# Project paths
# -------------------------
import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
RAW_DIR = PROJECT_DIR / "data" / "interim"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
RESULTS_DIR = PROJECT_DIR / "results"
LOGS_DIR = PROJECT_DIR / "logs"

for d in [PROCESSED_DIR / "integrated", METADATA_DIR, FIGURES_DIR, RESULTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INPUT_H5AD = PROCESSED_DIR / "integrated" / "GBM_core_4datasets_fullgene_postQC_merged_counts.h5ad"
OUTPUT_H5AD = PROCESSED_DIR / "integrated" / "GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad"

AUDIT_TSV = METADATA_DIR / "merged_metadata_audit_4datasets.tsv"
MISSING_TSV = METADATA_DIR / "merged_metadata_missing_values_4datasets.tsv"
COUNTS_SAMPLE_TSV = METADATA_DIR / "merged_metadata_cell_counts_by_sample_patient_condition.tsv"
COUNTS_DATASET_TSV = METADATA_DIR / "merged_metadata_cell_counts_by_dataset_condition.tsv"
OBS_COLUMNS_TSV = METADATA_DIR / "merged_metadata_obs_columns_after_standardization.tsv"
SCVI_KEYS_TSV = METADATA_DIR / "merged_metadata_recommended_scVI_keys.tsv"

MASTER_METADATA_CANDIDATES = [
    METADATA_DIR / "master_dataset_metadata_attach_ready_v4.tsv",
    METADATA_DIR / "master_dataset_metadata_attach_ready.tsv",
]

print("Input:", INPUT_H5AD, "| exists:", INPUT_H5AD.exists())
print("Output:", OUTPUT_H5AD)

if not INPUT_H5AD.exists():
    raise FileNotFoundError(f"Input merged h5ad not found: {INPUT_H5AD}")

In [ ]:
# -------------------------
# Utility functions
# -------------------------

def replace_file(path: Path):
    path = Path(path)
    if path.exists():
        path.unlink()
        print(f"[REPLACED] removed existing file: {path}")

def write_tsv_replace(df: pd.DataFrame, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    replace_file(path)
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} | shape={df.shape}")

def write_h5ad_replace(adata, path: Path, compression="gzip"):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    replace_file(path)
    adata.write_h5ad(path, compression=compression)
    print(f"[SAVED] {path}")

def clean_string_series(s):
    s = pd.Series(s).copy()
    s = s.astype("string")
    s = s.str.strip()
    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "NaN": pd.NA,
        "None": pd.NA,
        "none": pd.NA,
        "NA": pd.NA,
        "N/A": pd.NA,
        "null": pd.NA,
        "NULL": pd.NA,
    })
    return s

def coalesce_columns(df, candidates, default=None):
    out = pd.Series([pd.NA] * len(df), index=df.index, dtype="string")
    used = []
    for c in candidates:
        if c in df.columns:
            used.append(c)
            vals = clean_string_series(df[c])
            out = out.combine_first(vals)
    if default is not None:
        out = out.fillna(default)
    return out, used

def infer_geo_from_text(text):
    text = str(text)
    known = {
        "DS1_GSE103224": "GSE103224",
        "DS2_GSE141383": "GSE141383",
        "DS3_GSE182109": "GSE182109",
        "DS4_GSE278450": "GSE278450",
    }
    for core, geo in known.items():
        if core in text or geo in text:
            return geo
    return pd.NA

def infer_core_dataset_from_text(text):
    text = str(text)
    known = ["DS1_GSE103224", "DS2_GSE141383", "DS3_GSE182109", "DS4_GSE278450"]
    for core in known:
        if core in text:
            return core
    for core in known:
        geo = core.split("_", 1)[1]
        if geo in text:
            return core
    return pd.NA

def normalize_condition_value(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip()
    low = s.lower().replace("-", "").replace("_", "")
    if low in ["gbm", "glioblastoma", "glioblastomamultiforme", "idhwtgbm", "gbmidhwt"]:
        return "GBM"
    if low in ["ndgbm", "newlydiagnosedgbm", "primarygbm", "primary"]:
        return "ndGBM"
    if low in ["rgbm", "recurrentgbm", "recurrent"]:
        return "rGBM"
    if low in ["lgg", "lowergradeglioma", "lowgradeglioma"]:
        return "LGG"
    return s

def infer_tumor_status_from_condition(condition):
    c = str(condition).strip().lower()
    if c == "ndgbm":
        return "newly_diagnosed"
    if c == "rgbm":
        return "recurrent"
    if c == "lgg":
        return "lower_grade_glioma"
    if c == "gbm":
        return "gbm_unspecified"
    if c in ["unknown", "nan", "none", "", "<na>"]:
        return "unknown"
    return "other_or_unspecified"

def infer_diagnosis_from_condition(condition):
    c = str(condition).strip()
    if c in ["GBM", "ndGBM", "rGBM"]:
        return "glioblastoma"
    if c == "LGG":
        return "lower_grade_glioma"
    if c in ["unknown", "nan", "None", "", "<NA>"]:
        return "unknown"
    return c

def safe_make_category(obs, columns):
    for col in columns:
        if col in obs.columns:
            obs[col] = clean_string_series(obs[col]).fillna("unknown").astype(str).astype("category")
    return obs

def is_integer_like_sparse_or_array(x, max_check=100000):
    # Quick integer-like check without densifying huge matrices.
    try:
        import scipy.sparse as sp
        if sp.issparse(x):
            data = x.data
            if data.size == 0:
                return True
            sample = data[:min(max_check, data.size)]
        else:
            arr = np.asarray(x)
            sample = arr.ravel()[:min(max_check, arr.size)]
        if len(sample) == 0:
            return True
        return bool(np.allclose(sample, np.round(sample)))
    except Exception as e:
        print("[WARNING] integer-like check failed:", e)
        return None

In [ ]:
# -------------------------
# Read merged full-gene object
# -------------------------

adata = sc.read_h5ad(INPUT_H5AD)

print(adata)
print("n_obs:", adata.n_obs)
print("n_vars:", adata.n_vars)
print("layers:", list(adata.layers.keys()))
print("obsm:", list(adata.obsm.keys()))
print("obs columns:", len(adata.obs.columns))
print(list(adata.obs.columns)[:120])
print("var columns:", list(adata.var.columns)[:80])

if adata.n_vars < 10000:
    raise ValueError(
        f"This object has only {adata.n_vars} genes. "
        "For this restarted pipeline, the merged full-gene object should have ~15k common genes."
    )

if "counts" not in adata.layers:
    print("[WARNING] No counts layer found. If adata.X is raw counts, it will be copied to layers['counts'].")
    adata.layers["counts"] = adata.X.copy()

print("counts layer shape:", adata.layers["counts"].shape)
print("counts layer integer-like sample:", is_integer_like_sparse_or_array(adata.layers["counts"]))

In [ ]:
# -------------------------
# Build standardized metadata columns
# -------------------------

obs = adata.obs.copy()
original_obs_columns = list(obs.columns)

# Keep all original metadata columns; add standardized columns below.

# 1) core_dataset
core_dataset, used_core = coalesce_columns(
    obs,
    ["core_dataset", "dataset", "dataset_id", "study_id", "batch", "source_dataset", "orig.ident"],
)
if core_dataset.isna().any():
    inferred_from_index = pd.Series(obs.index, index=obs.index).map(infer_core_dataset_from_text).astype("string")
    core_dataset = core_dataset.combine_first(inferred_from_index)

if core_dataset.isna().any():
    geo_existing, _ = coalesce_columns(obs, ["geo_id", "GEO", "study"], default=pd.NA)
    inferred_from_geo = geo_existing.map(infer_core_dataset_from_text).astype("string")
    core_dataset = core_dataset.combine_first(inferred_from_geo)

core_dataset = clean_string_series(core_dataset).fillna("unknown_core_dataset")
obs["core_dataset"] = core_dataset.astype(str)

# 2) geo_id
geo_id, used_geo = coalesce_columns(obs, ["geo_id", "GEO", "study", "source_dataset", "dataset_geo"], default=pd.NA)
if geo_id.isna().any():
    inferred_from_core = obs["core_dataset"].map(infer_geo_from_text).astype("string")
    geo_id = geo_id.combine_first(inferred_from_core)
if geo_id.isna().any():
    inferred_from_index = pd.Series(obs.index, index=obs.index).map(infer_geo_from_text).astype("string")
    geo_id = geo_id.combine_first(inferred_from_index)

obs["geo_id"] = clean_string_series(geo_id).fillna("unknown_geo").astype(str)

# 3) dataset_id / study_id
dataset_id, used_dataset = coalesce_columns(
    obs,
    ["dataset_id", "dataset", "core_dataset", "source_dataset", "study_id", "orig.ident"],
)
dataset_id = clean_string_series(dataset_id).combine_first(obs["core_dataset"].astype("string"))
obs["dataset_id"] = clean_string_series(dataset_id).fillna(obs["core_dataset"].astype(str)).astype(str)
obs["study_id"] = obs["geo_id"].astype(str)

# 4) sample_id
sample_id, used_sample = coalesce_columns(
    obs,
    [
        "sample_id",
        "sample_id_global",
        "sample",
        "sample_name",
        "gsm_id",
        "GSM",
        "orig.ident",
        "library_id",
        "donor_sample",
    ],
)
sample_id = clean_string_series(sample_id)
sample_id = sample_id.fillna(obs["core_dataset"].astype(str) + "_sample_unknown")
obs["sample_id"] = sample_id.astype(str)

# 5) patient_id
patient_id, used_patient = coalesce_columns(
    obs,
    [
        "patient_id",
        "patient_id_global",
        "patient",
        "donor",
        "subject",
        "case_id",
        "tumor_id",
        "individual",
    ],
)
patient_id = clean_string_series(patient_id)
patient_id = patient_id.fillna(obs["sample_id"].astype(str))
obs["patient_id"] = patient_id.astype(str)

# 6) condition / diagnosis / tumor status
condition, used_condition = coalesce_columns(
    obs,
    [
        "condition",
        "tumor_status",
        "diagnosis",
        "disease",
        "disease_stage",
        "analysis_group_detail",
    ],
)
condition = clean_string_series(condition).map(normalize_condition_value)
condition = clean_string_series(condition).fillna("unknown")
obs["condition"] = condition.astype(str)

diagnosis, used_diagnosis = coalesce_columns(obs, ["diagnosis", "disease", "histology"], default=pd.NA)
diagnosis = clean_string_series(diagnosis)
diagnosis = diagnosis.combine_first(obs["condition"].map(infer_diagnosis_from_condition).astype("string"))
obs["diagnosis"] = clean_string_series(diagnosis).fillna("unknown").astype(str)

disease_stage, used_stage = coalesce_columns(obs, ["disease_stage", "stage", "grade", "tumor_grade"], default=pd.NA)
disease_stage = clean_string_series(disease_stage).fillna(obs["condition"].astype(str))
obs["disease_stage"] = disease_stage.astype(str)

tumor_status, used_status = coalesce_columns(obs, ["tumor_status", "recurrence_status", "treatment_status"], default=pd.NA)
tumor_status = clean_string_series(tumor_status)
tumor_status = tumor_status.combine_first(obs["condition"].map(infer_tumor_status_from_condition).astype("string"))
obs["tumor_status"] = clean_string_series(tumor_status).fillna("unknown").astype(str)

# 7) platform / technology
platform, used_platform = coalesce_columns(
    obs,
    ["platform", "technology", "chemistry", "assay_type", "sequencing_platform"],
    default=pd.NA,
)
obs["platform"] = clean_string_series(platform).fillna("unknown_platform").astype(str)

technology, used_technology = coalesce_columns(
    obs,
    ["technology", "assay_type", "chemistry", "platform"],
    default=pd.NA,
)
obs["technology"] = clean_string_series(technology).fillna(obs["platform"].astype(str)).astype(str)

# 8) batch columns for downstream scVI and bias checks
obs["study_batch"] = obs["core_dataset"].astype(str)
obs["sample_batch"] = obs["sample_id"].astype(str)
obs["scvi_batch"] = obs["sample_id"].astype(str)

# 9) source_file if possible
source_file, used_source = coalesce_columns(obs, ["source_file", "file", "input_file", "h5ad_file"], default=pd.NA)
source_file = clean_string_series(source_file).fillna(obs["core_dataset"].astype(str) + "_postQC.h5ad")
obs["source_file"] = source_file.astype(str)

# 10) analysis labels
analysis_group_major, used_major = coalesce_columns(
    obs,
    ["analysis_group_major", "analysis_group", "group_major"],
    default=pd.NA,
)
obs["analysis_group_major"] = clean_string_series(analysis_group_major).fillna("core").astype(str)

analysis_group_detail, used_detail = coalesce_columns(
    obs,
    ["analysis_group_detail", "group_detail"],
    default=pd.NA,
)
obs["analysis_group_detail"] = clean_string_series(analysis_group_detail).fillna(obs["condition"].astype(str)).astype(str)

obs["ready_for_integration"] = "yes"

# 11) cell ID trace
if "cell_id_original" not in obs.columns:
    obs["cell_id_original"] = adata.obs_names.astype(str)
if not adata.obs_names.is_unique:
    print("[WARNING] obs_names were not unique. Making them unique.")
    adata.obs_names_make_unique()

# Apply standardized obs back to object
adata.obs = obs

used_map = pd.DataFrame([
    {"standard_column": "core_dataset", "source_columns_used": " | ".join(used_core)},
    {"standard_column": "geo_id", "source_columns_used": " | ".join(used_geo)},
    {"standard_column": "dataset_id", "source_columns_used": " | ".join(used_dataset)},
    {"standard_column": "sample_id", "source_columns_used": " | ".join(used_sample)},
    {"standard_column": "patient_id", "source_columns_used": " | ".join(used_patient)},
    {"standard_column": "condition", "source_columns_used": " | ".join(used_condition)},
    {"standard_column": "diagnosis", "source_columns_used": " | ".join(used_diagnosis)},
    {"standard_column": "disease_stage", "source_columns_used": " | ".join(used_stage)},
    {"standard_column": "tumor_status", "source_columns_used": " | ".join(used_status)},
    {"standard_column": "platform", "source_columns_used": " | ".join(used_platform)},
    {"standard_column": "technology", "source_columns_used": " | ".join(used_technology)},
    {"standard_column": "source_file", "source_columns_used": " | ".join(used_source)},
    {"standard_column": "analysis_group_major", "source_columns_used": " | ".join(used_major)},
    {"standard_column": "analysis_group_detail", "source_columns_used": " | ".join(used_detail)},
])
display(used_map)

In [ ]:
# -------------------------
# Optional: enrich from master sample metadata if an appropriate file exists
# -------------------------
# Conservative: only fills missing/unknown standardized fields by sample_id.
# It never overwrites existing informative per-cell metadata.

master_path = next((p for p in MASTER_METADATA_CANDIDATES if p.exists()), None)

if master_path is None:
    print("[INFO] No master metadata file found. Skipping optional sample-level enrichment.")
else:
    print("[INFO] Found master metadata:", master_path)
    master = pd.read_csv(master_path, sep="\t", dtype=str)
    for c in master.columns:
        master[c] = clean_string_series(master[c])

    if "sample_id_global" in master.columns:
        fallback_sample = master["sample_id"] if "sample_id" in master.columns else pd.Series(pd.NA, index=master.index)
        master["_sample_id_std"] = master["sample_id_global"].combine_first(fallback_sample)
    elif "sample_id" in master.columns:
        master["_sample_id_std"] = master["sample_id"]
    elif "gsm_id" in master.columns:
        master["_sample_id_std"] = master["gsm_id"]
    else:
        master["_sample_id_std"] = pd.NA

    if "patient_id_global" in master.columns:
        fallback_patient = master["patient_id"] if "patient_id" in master.columns else pd.Series(pd.NA, index=master.index)
        master["_patient_id_std"] = master["patient_id_global"].combine_first(fallback_patient)
    elif "patient_id" in master.columns:
        master["_patient_id_std"] = master["patient_id"]
    else:
        master["_patient_id_std"] = pd.NA

    master["_sample_id_std"] = clean_string_series(master["_sample_id_std"])
    master_use = master.dropna(subset=["_sample_id_std"]).drop_duplicates("_sample_id_std").copy()
    sample_to_master = master_use.set_index("_sample_id_std")

    enrich_cols = {
        "patient_id": "_patient_id_std",
        "condition": "condition",
        "diagnosis": "diagnosis",
        "platform": "platform",
        "technology": "assay_type",
        "tumor_status": "recurrence_status",
        "disease_stage": "tumor_grade",
    }

    filled = []
    for target_col, master_col in enrich_cols.items():
        if master_col not in sample_to_master.columns or target_col not in adata.obs.columns:
            continue

        mapped = adata.obs["sample_id"].astype(str).map(sample_to_master[master_col].astype("string"))
        mapped = clean_string_series(mapped)
        old = clean_string_series(adata.obs[target_col])
        old_missing = old.isna() | old.isin(["unknown", "unknown_platform"])
        fill_mask = old_missing & mapped.notna()

        n_fill = int(fill_mask.sum())
        if n_fill > 0:
            adata.obs.loc[fill_mask, target_col] = mapped.loc[fill_mask].astype(str).values

        filled.append({"target_col": target_col, "master_col": master_col, "n_cells_filled": n_fill})

    enrich_report = pd.DataFrame(filled)
    display(enrich_report)

In [ ]:
# -------------------------
# Final cleaning / categorical conversion
# -------------------------

standard_columns = [
    "core_dataset",
    "dataset_id",
    "geo_id",
    "study_id",
    "sample_id",
    "patient_id",
    "condition",
    "diagnosis",
    "disease_stage",
    "tumor_status",
    "platform",
    "technology",
    "study_batch",
    "sample_batch",
    "scvi_batch",
    "source_file",
    "analysis_group_major",
    "analysis_group_detail",
    "ready_for_integration",
]

for col in standard_columns:
    if col not in adata.obs.columns:
        adata.obs[col] = "unknown"
    adata.obs[col] = clean_string_series(adata.obs[col]).fillna("unknown").astype(str)

adata.obs["condition"] = adata.obs["condition"].map(normalize_condition_value)
adata.obs["condition"] = clean_string_series(adata.obs["condition"]).fillna("unknown").astype(str)

diagnosis_missing = adata.obs["diagnosis"].astype(str).isin(["unknown", "nan", "None", "<NA>", ""])
adata.obs.loc[diagnosis_missing, "diagnosis"] = adata.obs.loc[diagnosis_missing, "condition"].map(infer_diagnosis_from_condition).astype(str)

status_missing = adata.obs["tumor_status"].astype(str).isin(["unknown", "nan", "None", "<NA>", ""])
adata.obs.loc[status_missing, "tumor_status"] = adata.obs.loc[status_missing, "condition"].map(infer_tumor_status_from_condition).astype(str)

categorical_cols = standard_columns.copy()
adata.obs = safe_make_category(adata.obs, categorical_cols)

adata.uns["metadata_standardization_step"] = {
    "created": datetime.now().isoformat(timespec="seconds"),
    "input_h5ad": str(INPUT_H5AD),
    "output_h5ad": str(OUTPUT_H5AD),
    "n_obs": int(adata.n_obs),
    "n_vars": int(adata.n_vars),
    "counts_layer_preserved": bool("counts" in adata.layers),
    "standard_columns": standard_columns,
    "recommended_scvi_batch_key": "scvi_batch",
    "recommended_scvi_categorical_covariates": ["core_dataset", "condition", "patient_id"],
}

print(adata)
print("standard columns:")
print(standard_columns)

In [ ]:
# -------------------------
# Metadata audit reports
# -------------------------

audit_rows = []

for col in standard_columns:
    s = adata.obs[col].astype(str)
    audit_rows.append({
        "column": col,
        "present": col in adata.obs.columns,
        "n_unique": int(s.nunique(dropna=False)),
        "n_missing_or_unknown": int(s.isin(["", "nan", "None", "<NA>", "unknown"]).sum()),
        "example_values": " | ".join(s.value_counts(dropna=False).head(12).index.astype(str).tolist()),
    })

audit_rows.extend([
    {
        "column": "_object_n_cells",
        "present": True,
        "n_unique": int(adata.n_obs),
        "n_missing_or_unknown": 0,
        "example_values": str(adata.n_obs),
    },
    {
        "column": "_object_n_genes",
        "present": True,
        "n_unique": int(adata.n_vars),
        "n_missing_or_unknown": 0,
        "example_values": str(adata.n_vars),
    },
    {
        "column": "_has_counts_layer",
        "present": True,
        "n_unique": int("counts" in adata.layers),
        "n_missing_or_unknown": 0,
        "example_values": str("counts" in adata.layers),
    },
])

audit = pd.DataFrame(audit_rows)
write_tsv_replace(audit, AUDIT_TSV)
display(audit)

missing_rows = []
for col in standard_columns:
    s = adata.obs[col].astype(str)
    vc = s.value_counts(dropna=False)
    missing_rows.append({
        "column": col,
        "n_cells": int(len(s)),
        "n_missing_empty_or_unknown": int(s.isin(["", "nan", "None", "<NA>", "unknown"]).sum()),
        "fraction_missing_empty_or_unknown": float(s.isin(["", "nan", "None", "<NA>", "unknown"]).mean()),
        "n_unique": int(s.nunique(dropna=False)),
        "top_value": str(vc.index[0]) if len(vc) else "",
        "top_value_n_cells": int(vc.iloc[0]) if len(vc) else 0,
        "top_value_fraction": float(vc.iloc[0] / len(s)) if len(vc) else 0.0,
    })

missing = pd.DataFrame(missing_rows)
write_tsv_replace(missing, MISSING_TSV)
display(missing)

In [ ]:
# -------------------------
# Cell count reports
# -------------------------

counts_sample = (
    adata.obs
    .groupby(
        ["core_dataset", "geo_id", "sample_id", "patient_id", "condition", "diagnosis", "tumor_status"],
        observed=True
    )
    .size()
    .reset_index(name="n_cells")
    .sort_values(["core_dataset", "sample_id", "patient_id"])
)
write_tsv_replace(counts_sample, COUNTS_SAMPLE_TSV)
display(counts_sample.head(50))

counts_dataset = (
    adata.obs
    .groupby(["core_dataset", "geo_id", "condition", "diagnosis", "tumor_status"], observed=True)
    .size()
    .reset_index(name="n_cells")
    .sort_values(["core_dataset", "condition"])
)
write_tsv_replace(counts_dataset, COUNTS_DATASET_TSV)
display(counts_dataset)

obs_cols_report = pd.DataFrame({
    "obs_column": list(adata.obs.columns),
    "dtype": [str(adata.obs[c].dtype) for c in adata.obs.columns],
    "n_unique": [int(adata.obs[c].astype(str).nunique(dropna=False)) for c in adata.obs.columns],
})
write_tsv_replace(obs_cols_report, OBS_COLUMNS_TSV)
display(obs_cols_report.head(80))

In [ ]:
# -------------------------
# Recommended downstream keys
# -------------------------

recommended = pd.DataFrame([
    {
        "purpose": "scVI batch_key",
        "recommended_key": "scvi_batch",
        "reason": "sample-level batch is conservative for multi-study integration",
        "n_unique": int(adata.obs["scvi_batch"].astype(str).nunique()),
    },
    {
        "purpose": "scVI categorical_covariate_key",
        "recommended_key": "core_dataset",
        "reason": "study/dataset source must be retained for technical/bias tracking",
        "n_unique": int(adata.obs["core_dataset"].astype(str).nunique()),
    },
    {
        "purpose": "scVI categorical_covariate_key",
        "recommended_key": "condition",
        "reason": "GBM/LGG/newly diagnosed/recurrent status should not be lost",
        "n_unique": int(adata.obs["condition"].astype(str).nunique()),
    },
    {
        "purpose": "bias analysis",
        "recommended_key": "patient_id",
        "reason": "needed for single-patient dominant cluster detection",
        "n_unique": int(adata.obs["patient_id"].astype(str).nunique()),
    },
    {
        "purpose": "bias analysis",
        "recommended_key": "core_dataset",
        "reason": "needed for dataset-biased cluster detection",
        "n_unique": int(adata.obs["core_dataset"].astype(str).nunique()),
    },
])

write_tsv_replace(recommended, SCVI_KEYS_TSV)
display(recommended)

In [ ]:
# -------------------------
# Final safety checks before saving
# -------------------------

required_for_next_steps = [
    "core_dataset",
    "dataset_id",
    "geo_id",
    "sample_id",
    "patient_id",
    "condition",
    "diagnosis",
    "platform",
    "technology",
    "scvi_batch",
]

missing_required = [c for c in required_for_next_steps if c not in adata.obs.columns]
if missing_required:
    raise KeyError(f"Missing required obs columns: {missing_required}")

if adata.n_vars < 10000:
    raise ValueError(f"Unexpectedly low gene count before saving: {adata.n_vars}")

if "counts" not in adata.layers:
    raise KeyError("counts layer missing before saving.")

if not adata.obs_names.is_unique:
    raise ValueError("obs_names are not unique.")

summary = {
    "n_cells": int(adata.n_obs),
    "n_genes": int(adata.n_vars),
    "n_core_datasets": int(adata.obs["core_dataset"].astype(str).nunique()),
    "n_samples": int(adata.obs["sample_id"].astype(str).nunique()),
    "n_patients": int(adata.obs["patient_id"].astype(str).nunique()),
    "conditions": sorted(adata.obs["condition"].astype(str).unique().tolist()),
    "has_counts_layer": bool("counts" in adata.layers),
}
print(json.dumps(summary, indent=2, ensure_ascii=False))

In [ ]:
# -------------------------
# Save metadata-standardized full-gene object
# -------------------------

write_h5ad_replace(adata, OUTPUT_H5AD)

print("\nDONE.")
print("Use this file for the next HVG/scVI branch:")
print(OUTPUT_H5AD)

## Expected next input

After this notebook runs successfully, the next notebook should use:

`${GLIOMA_PROJECT_ROOT}/processed/integrated/GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad`

The next step is HVG/scVI branch creation from this metadata-standardized full-gene object.